In [ ]:
# Copyright (c) 2025 Microsoft Corporation.

import sys

sys.path.insert(1, "../../../")

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import logging
import os

import pandas as pd
import tiktoken
from pydantic import SecretStr

from benchmark_qed.autod.data_processor.embedding import TextEmbedder
from benchmark_qed.autod.io.text_unit import load_text_units
from benchmark_qed.autoq.io.question import (
    save_questions,
)
from benchmark_qed.config.llm_config import LLMConfig, LLMProvider
from benchmark_qed.llm.factory import ModelFactory

logging.basicConfig(level=logging.INFO)

if logging.getLogger("httpx") is not None:
    logging.getLogger("httpx").setLevel(logging.ERROR)

# Experiment 11b: AutoQ Data-Local Question Generation — Podcast Dataset

This notebook demonstrates data-local question generation using AutoQ on the **Behind the Tech Podcast Transcripts** dataset. Data-local questions target specific details grounded in particular text excerpts from the corpus.

The generation pipeline consists of:
1. **Data Sampling**: Load and chunk podcast transcripts into text units, embed them, and create a clustered sample using AutoD.
2. **Data-Local Question Generation**: For each cluster of sample texts, generate candidate questions using a two-step (extract + expand) process, then rank and select the best questions.
3. **Assertion Generation** *(optional)*: Generate testable factual statements for each selected question to enable assertion-based answer accuracy evaluation.

The output questions can be used directly as evaluation queries for RAG systems, or passed to AutoE for answer comparison and scoring.

## Configs

In [ ]:
# DATA CONFIGS
INPUT_DATA_PATH = "../../datasets/podcast/raw_data"
OUTPUT_DATA_PATH = "../../output/podcast/processed_data"
OUTPUT_QUESTIONS_PATH = "../../output/podcast/questions"
TEXT_COLUMN = "text"
METADATA_COLUMNS = ["episode"]
FILE_ENCODING = "utf-8-sig"

# tokenizer used for chunking documents into text units
ENCODING_MODEL = "o200k_base"
CHUNK_SIZE = 600
CHUNK_OVERLAP = 100

# DATA SAMPLING CONFIGS
# These configs control the breadth and depth of the selected data sample.
# Adjust these parameters based on your data size and the number of questions to be generated.
# The final sample size will be NUM_CLUSTERS * NUM_SAMPLES_PER_CLUSTER
NUM_CLUSTERS = 20
NUM_SAMPLES_PER_CLUSTER = 10
RANDOM_SEED = 42

# QUESTION GENERATION CONFIGS
NUM_QUESTIONS = 10  # Number of final questions to select
OVERSAMPLE_FACTOR = 2.0  # Factor by which to overgenerate candidate questions

In [ ]:
# MODEL CONFIGS
API_KEY = SecretStr(os.getenv("OPENAI_API_KEY", ""))
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4.1"
LLM_PARAMS = {
    "temperature": 0.0,
    "seed": 42,
}  # adjust this based on your model. For example, some reasoning models do not support temperature settings
CONCURRENT_REQUESTS = (
    8  # Control for request concurrency. Adjust this based on your model capacity.
)

# ASSERTION GENERATION CONFIGS
MAX_ASSERTIONS = 20  # Maximum number of assertions per question (set to 0 to disable)
ENABLE_VALIDATION = True  # Set to True to validate assertions against sources
MIN_VALIDATION_SCORE = 3  # Minimum score (1-5) for grounding, relevance, verifiability

# Parallelism for assertion generation (adjust based on your model rate limits)
CONCURRENT_LOCAL_QUESTIONS = 8  # Questions to process in parallel for local assertions

text_embedder = TextEmbedder(
    ModelFactory.create_embedding_model(
        LLMConfig(
            model=EMBEDDING_MODEL,
            api_key=API_KEY,
            llm_provider=LLMProvider.OpenAIEmbedding,
        )
    )
)
llm = ModelFactory.create_chat_model(
    model_config=LLMConfig(
        model=LLM_MODEL,
        api_key=API_KEY,
        llm_provider=LLMProvider.OpenAIChat,
        call_args=LLM_PARAMS,
    )
)
token_encoder = tiktoken.get_encoding(ENCODING_MODEL)

## Data Sampling

Load podcast transcripts from the input folder, chunk them into text units, embed all text units, and sample a clustered subset. These clustered sample texts ground the question generation process.

If you have previously run this sampling step, you can skip it and load the saved sample from disk in the next section.

In [ ]:
from benchmark_qed.autod.sampler.sample_gen import acreate_clustered_sample

clustered_sample = await acreate_clustered_sample(
    input_path=INPUT_DATA_PATH,
    output_path=OUTPUT_DATA_PATH,
    text_embedder=text_embedder,
    num_clusters=NUM_CLUSTERS,
    num_samples_per_cluster=NUM_SAMPLES_PER_CLUSTER,
    input_type="json",
    text_tag=TEXT_COLUMN,
    metadata_tags=METADATA_COLUMNS,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    file_encoding=FILE_ENCODING,
    token_encoding=ENCODING_MODEL,
    random_seed=RANDOM_SEED,
)
print(
    f"Sampled {len(clustered_sample.sample_texts)} samples from {len(clustered_sample.text_units)} text units in {len(clustered_sample.documents)} documents."
)

## Data-Local Question Generation

Generate questions grounded in specific text passages from the podcast transcripts. Each question is associated with a set of source text units and extracted claims that support it.

The generation process:
1. Groups sample texts by cluster
2. Generates candidate questions for each cluster using a two-step (extract + expand) chain
3. Ranks and selects the best `NUM_QUESTIONS` questions based on similarity and coverage metrics
4. Optionally generates assertions for the selected questions

In [ ]:
from benchmark_qed.autoq.config import (
    AssertionConfig,
    LocalAssertionConfig,
)
from benchmark_qed.autoq.question_gen.data_questions.local_question_gen import (
    DataLocalQuestionGen,
)

# Load clustered text sample from disk (result from the data sampling step above).
# You can also use clustered_sample.sample_texts directly if the sampling step was just run.
sample_texts_df = pd.read_parquet(f"{OUTPUT_DATA_PATH}/sample_texts.parquet")
sample_texts = load_text_units(df=sample_texts_df)

# Configure assertion generation
# Set max_assertions to an integer to limit assertions, 0 to disable, None for unlimited
local_assertion_config = LocalAssertionConfig(
    max_assertions=MAX_ASSERTIONS,
    enable_validation=ENABLE_VALIDATION,
    min_validation_score=MIN_VALIDATION_SCORE,
    concurrent_llm_calls=CONCURRENT_REQUESTS,
    max_concurrent_questions=CONCURRENT_LOCAL_QUESTIONS,
)
assertion_config = AssertionConfig(local=local_assertion_config)

data_local_generator = DataLocalQuestionGen(
    llm=llm,
    llm_params=LLM_PARAMS,
    text_embedder=text_embedder,
    text_units=sample_texts,
    concurrent_coroutines=CONCURRENT_REQUESTS,
    random_seed=RANDOM_SEED,
    assertion_config=assertion_config,
)

data_local_question_results = await data_local_generator.agenerate(
    num_questions=NUM_QUESTIONS,
    oversample_factor=OVERSAMPLE_FACTOR,
)

print(
    f"Generated {len(data_local_question_results.candidate_questions)} candidate questions."
)
print(
    f"Selected {len(data_local_question_results.selected_questions)} final questions."
)

In [ ]:
# Save both candidate questions and the final selected questions
save_questions(
    data_local_question_results.selected_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_local_questions/",
    "selected_questions",
)
save_questions(
    data_local_question_results.selected_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_local_questions/",
    "selected_questions_text",
    question_text_only=True,
)
save_questions(
    data_local_question_results.candidate_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_local_questions/",
    "candidate_questions",
)

print("Saved selected and candidate questions to disk.")
print("\nSelected questions:")
for i, q in enumerate(data_local_question_results.selected_questions):
    print(f"{i + 1}. {q.text}")